# Lab 4 — Robot Model, RViz, and Gazebo


> **Rewritten AIG240 Notebook Version**  
> This notebook turns the original web manual into a cleaner, notebook-friendly lab. It keeps the technical intent but reorganizes the workflow around learning goals, verification checkpoints, a ROS1 controller-package extension, troubleshooting, submission evidence, and reflection prompts.  
>
> **ROS version target:** ROS1 Melodic on Ubuntu 18.04, matching the JetAuto/Jetson Nano environment used in the course.  
> **Important:** Run ROS commands in a Linux terminal unless a cell explicitly says it is a Python helper/visualization cell.


<svg width="100%" viewBox="0 0 1050 150" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Simulation workflow">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1050" height="150" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="30" font-family="Arial" font-size="20" font-weight="700" fill="#0f172a">Simulation workflow</text>
<rect x="25.0" y="55" width="180.0" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="115.0" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">URDF/XACRO model</text>
<line x1="205.0" y1="84" x2="224.0" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="230.0" y="55" width="180.0" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="320.0" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">RViz visualization</text>
<line x1="410.0" y1="84" x2="429.0" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="435.0" y="55" width="180.0" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="525.0" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Gazebo physics</text>
<text x="525.0" y="97" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">world</text>
<line x1="615.0" y1="84" x2="634.0" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="640.0" y="55" width="180.0" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="730.0" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Controller node</text>
<line x1="820.0" y1="84" x2="839.0" y2="84" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="845.0" y="55" width="180.0" height="58" rx="12" fill="white" stroke="#64748b"/>
<text x="935.0" y="80" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">cmd_vel messages</text>
</svg>


## Learning outcomes
You will connect robot modeling concepts to simulation: URDF/XACRO describes the robot, RViz visualizes robot state and sensor data, and Gazebo simulates physics and environments.

## Concepts
- **URDF:** XML model of links, joints, visual geometry, collision geometry, and inertial properties.
- **XACRO:** macro-based XML that generates URDF; useful for reusable and parameterized robot descriptions.
- **RViz:** visualization/debugging tool for robot state, transforms, maps, and sensors.
- **Gazebo:** simulator with physics, worlds, robot plugins, and virtual sensors.


## Install common simulation packages
```bash
sudo apt update
sudo apt install -y gazebo9 ros-melodic-gazebo-ros-pkgs ros-melodic-gazebo-ros-control
sudo apt install -y ros-melodic-joint-state-publisher ros-melodic-robot-state-publisher ros-melodic-xacro
```


## Get or verify the JetAuto workspace
The course uses a JetAuto ROS workspace that contains model, Gazebo, controller, SLAM, and navigation packages. Verify the workspace exists:
```bash
ls ~/jetauto_ws/src
```
Then build if needed:
```bash
cd ~/jetauto_ws
catkin_make
source devel/setup.bash
```


## View the robot model
Typical workflow:
```bash
roslaunch jetauto_description display.launch
```
In RViz, inspect:
- robot links and joints
- TF frames
- laser/camera frames if available
- whether the model is visually aligned and stable


## Launch JetAuto in Gazebo
```bash
roslaunch jetauto_gazebo worlds.launch
```
Press **Play** in Gazebo if the simulation is paused. Keep an eye on the terminal for missing plugin or package errors.


## Publish a motion command in simulation
First stop the robot:
```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist '{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}'
```
Then move forward gently:
```bash
rostopic pub -1 /jetauto_controller/cmd_vel geometry_msgs/Twist '{linear: {x: 0.2, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}'
```
Always send a stop command after testing.


## Optional ROS1 keyboard controller package

In the previous step, you tested motion by publishing a `Twist` message manually. A more realistic workflow is to create a small ROS package that publishes velocity commands from keyboard input.

Create a package inside the JetAuto workspace:

```bash
cd ~/jetauto_ws/src
catkin_create_pkg lab4_jetauto_control rospy geometry_msgs
```

Create a scripts folder and a Python controller file:

```bash
cd ~/jetauto_ws/src/lab4_jetauto_control
mkdir -p scripts
nano scripts/teleop_key_control.py
chmod +x scripts/teleop_key_control.py
```

Suggested `teleop_key_control.py` structure:

```python
#!/usr/bin/env python

import sys
import termios
import tty

import rospy
from geometry_msgs.msg import Twist


MOVE_BINDINGS = {
    'w': (0.2, 0.0),    # forward
    's': (-0.2, 0.0),   # backward
    'a': (0.0, 0.6),    # rotate left
    'd': (0.0, -0.6),   # rotate right
    'x': (0.0, 0.0),    # stop
}


def get_key():
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key


def main():
    rospy.init_node('lab4_keyboard_controller')

    # If your JetAuto simulation uses a different command topic,
    # update this parameter when launching or edit the default below.
    cmd_topic = rospy.get_param('~cmd_topic', '/jetauto_controller/cmd_vel')
    pub = rospy.Publisher(cmd_topic, Twist, queue_size=10)

    print('Use W/S to move, A/D to rotate, X to stop, Q to quit.')

    try:
        while not rospy.is_shutdown():
            key = get_key().lower()
            if key == 'q':
                break

            twist = Twist()
            linear_x, angular_z = MOVE_BINDINGS.get(key, (0.0, 0.0))
            twist.linear.x = linear_x
            twist.angular.z = angular_z
            pub.publish(twist)

    finally:
        stop = Twist()
        pub.publish(stop)
        print('Robot stopped.')


if __name__ == '__main__':
    main()
```

Build and source the workspace:

```bash
cd ~/jetauto_ws
catkin_make
source devel/setup.bash
```

Run the controller while Gazebo is open:

```bash
rosrun lab4_jetauto_control teleop_key_control.py
```

Verification:

```bash
rostopic echo /jetauto_controller/cmd_vel
```

You should see `geometry_msgs/Twist` messages change when you press movement keys. Always stop the robot before closing the terminal.


## Lab task

Complete the ROS1 workflow in this lab and explain how the robot model, visualization, simulation, and velocity command pipeline connect.

Your explanation should describe the role of:

- URDF/XACRO
- `robot_state_publisher`
- TF frames
- RViz
- Gazebo
- `cmd_vel` / `geometry_msgs/Twist`
- the optional keyboard controller package, if you implemented it

## ROS1 submission checklist

Submit the following evidence:

- Screenshot of the robot model displayed in RViz.
- Screenshot of the JetAuto robot running in Gazebo.
- Terminal output showing `rostopic list`.
- Terminal output showing either `rostopic echo /jetauto_controller/cmd_vel` or a successful `rostopic pub` command.
- If you implemented the optional controller package, include:
  - the package name: `lab4_jetauto_control`
  - the script name: `teleop_key_control.py`
  - a screenshot or terminal output showing the controller running
  - a short note explaining which topic it publishes to
- A short written answer explaining how URDF/XACRO, RViz, Gazebo, TF, and `cmd_vel` work together in the simulation stack.

## Debugging checklist

- If the robot does not move, confirm the command topic using `rostopic list`.
- If the topic name is different, update the publisher topic in your command or controller script.
- If Gazebo opens with an empty world, check the launch file path and package name.
- If RViz frames are missing, inspect `/tf` and confirm that `robot_state_publisher` is running.
- If `rosrun lab4_jetauto_control teleop_key_control.py` fails, confirm the script is executable with `chmod +x`.
- If Python import errors appear, rebuild and source the workspace again:

```bash
cd ~/jetauto_ws
catkin_make
source devel/setup.bash
```


---
# Appendix A — ROS2 Version of This Lab

## A.1 ROS1 ↔ ROS2 command translation

| Goal | ROS1 command | ROS2 command |
|---|---|---|
| Source environment | `source devel/setup.bash` | `source install/setup.bash` |
| Build workspace | `catkin_make` | `colcon build --symlink-install` |
| List topics | `rostopic list` | `ros2 topic list` |
| Echo a topic | `rostopic echo /topic` | `ros2 topic echo /topic` |
| List nodes | `rosnode list` | `ros2 node list` |
| Launch a file | `roslaunch pkg file.launch` | `ros2 launch pkg file.launch.py` |
| Run a node | `rosrun pkg node` | `ros2 run pkg node` |
| Publish velocity | `rostopic pub ...` | `ros2 topic pub ...` |
| Visualizer | `rviz` | `rviz2` |

Key conceptual difference: ROS2 does **not** use `roscore`. Discovery is handled by DDS middleware, so if nodes cannot see each other, check environment sourcing, `ROS_DOMAIN_ID`, network settings, and whether all terminals are using the same ROS2 distribution.


## A.2 Install the ROS2 simulation tools

For ROS2 Jazzy on Ubuntu 24.04:

```bash
sudo apt update
sudo apt install -y ros-jazzy-desktop ros-dev-tools
sudo apt install -y ros-jazzy-joint-state-publisher-gui ros-jazzy-robot-state-publisher ros-jazzy-xacro
sudo apt install -y ros-jazzy-ros-gz
```

For ROS2 Humble on Ubuntu 22.04, replace `jazzy` with `humble`:

```bash
sudo apt update
sudo apt install -y ros-humble-desktop ros-dev-tools
sudo apt install -y ros-humble-joint-state-publisher-gui ros-humble-robot-state-publisher ros-humble-xacro
sudo apt install -y ros-humble-ros-gz
```

Source ROS2 in every new terminal:

```bash
source /opt/ros/jazzy/setup.bash      # Ubuntu 24.04 / Jazzy
# source /opt/ros/humble/setup.bash   # Ubuntu 22.04 / Humble
```


## A.3 Create a ROS2 workspace

```bash
mkdir -p ~/lab4_ws/src
cd ~/lab4_ws
colcon build --symlink-install
source install/setup.bash
```

A clean ROS2 workspace should contain an `install/`, `build/`, and `log/` folder after building. If `colcon` is missing, install the ROS development tools for your distribution.

Verification:

```bash
ros2 --help
ros2 topic list
ros2 node list
```


## A.4 Robot model and RViz2 workflow

If your ROS2 package already contains a URDF/XACRO model, generate and inspect the robot description:

```bash
cd ~/lab4_ws
source /opt/ros/jazzy/setup.bash
source install/setup.bash

# Example pattern; update package and file names for your robot.
ros2 run xacro xacro src/<robot_description_pkg>/urdf/<robot>.urdf.xacro > /tmp/<robot>.urdf
check_urdf /tmp/<robot>.urdf
```

Publish the robot state and open RViz2:

```bash
ros2 run robot_state_publisher robot_state_publisher /tmp/<robot>.urdf
rviz2
```

In RViz2, add or verify these displays:

1. **RobotModel** — confirms that the URDF/XACRO loads correctly.
2. **TF** — confirms that frames are being published.
3. **LaserScan / Image / PointCloud2** — optional, depending on the sensors available in the model.

Checkpoint screenshot: include one RViz2 screenshot showing the robot model and TF tree.


## A.5 Gazebo Sim workflow in ROS2

For modern ROS2, prefer **Gazebo Sim** integration through `ros_gz` rather than old Gazebo Classic packages.

Basic launch pattern:

```bash
# Start Gazebo with an empty world
ros2 launch ros_gz_sim gz_sim.launch.py gz_args:="-r empty.sdf"
```

Spawn a robot from a URDF file:

```bash
ros2 run ros_gz_sim create \
  -name lab4_robot \
  -file /tmp/<robot>.urdf \
  -x 0 -y 0 -z 0.2
```

After spawning, verify that the simulation is running:

```bash
ros2 topic list
ros2 node list
```

If the robot appears but does not move, the model may be missing the required Gazebo/ROS2 control plugins, differential-drive plugin, or topic bridge. In that case, focus the lab submission on model visualization, TF inspection, and topic verification unless your instructor provides a ROS2-ready robot package.


## A.6 Publish velocity commands in ROS2

ROS1 uses `rostopic pub`; ROS2 uses `ros2 topic pub`. The message type is still `geometry_msgs/msg/Twist`, but the type syntax changes.

Publish a single stop command:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist \
"{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

Publish a gentle forward command at 2 Hz:

```bash
ros2 topic pub --rate 2 /cmd_vel geometry_msgs/msg/Twist \
"{linear: {x: 0.2, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

Stop the robot after testing:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist \
"{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

If your robot uses a namespaced topic, replace `/cmd_vel` with the correct topic from:

```bash
ros2 topic list | grep cmd
```


## A.7 Optional ROS2 keyboard controller package

Create a Python ROS2 package:

```bash
cd ~/lab4_ws/src
ros2 pkg create lab4_jetauto_control_ros2 --build-type ament_python --dependencies rclpy geometry_msgs
```

Example `teleop_key_control.py` structure:

```python
#!/usr/bin/env python3

import sys
import termios
import tty

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist


class SimpleTeleop(Node):
    def __init__(self):
        super().__init__('simple_teleop')
        self.publisher = self.create_publisher(Twist, '/cmd_vel', 10)
        self.speed = 0.2
        self.turn = 0.6

    def publish_cmd(self, linear_x: float, angular_z: float) -> None:
        msg = Twist()
        msg.linear.x = linear_x
        msg.angular.z = angular_z
        self.publisher.publish(msg)


def get_key() -> str:
    old_settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, old_settings)
    return key


def main():
    rclpy.init()
    node = SimpleTeleop()
    print('Use w/s for forward/backward, a/d for turning, space to stop, q to quit.')

    try:
        while rclpy.ok():
            key = get_key()
            if key == 'w':
                node.publish_cmd(node.speed, 0.0)
            elif key == 's':
                node.publish_cmd(-node.speed, 0.0)
            elif key == 'a':
                node.publish_cmd(0.0, node.turn)
            elif key == 'd':
                node.publish_cmd(0.0, -node.turn)
            elif key == ' ':
                node.publish_cmd(0.0, 0.0)
            elif key == 'q':
                break
    finally:
        node.publish_cmd(0.0, 0.0)
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()
```

Make it executable and rebuild:

```bash
chmod +x ~/lab4_ws/src/lab4_jetauto_control_ros2/lab4_jetauto_control_ros2/teleop_key_control.py
cd ~/lab4_ws
colcon build --symlink-install
source install/setup.bash
ros2 run lab4_jetauto_control_ros2 teleop_key_control.py
```

Depending on how the package entry points are configured, your instructor may ask you to add this script to `setup.py` before using `ros2 run`.


## A.8 ROS2 submission checklist

Students using ROS2 should submit the same evidence as ROS1 students, with ROS2-equivalent commands:

- Screenshot of the robot model in RViz2.
- Screenshot of the robot or empty world in Gazebo Sim.
- Terminal output showing `ros2 topic list`.
- Terminal output showing either `ros2 topic echo /cmd_vel` or a successful `ros2 topic pub` command.
- A short explanation of how URDF/XACRO, `robot_state_publisher`, TF, RViz2, Gazebo Sim, and `cmd_vel` connect in the ROS2 workflow.
- One troubleshooting note: describe a problem encountered and the command used to diagnose it.
